In [ ]:
import pandas as pd
df = pd.read_csv("retrievable_peptides.tsv", sep="\t")

# Keep rows with combined_n_noncontained_le9 >= 2
if "combined_n_noncontained_le9" in df.columns:
    df = df[pd.to_numeric(df["combined_n_noncontained_le9"], errors="coerce").ge(2)].copy()

# Keep rows where In_2.1.22-only is --- / empty / NA
if "In_2.1.22-only" in df.columns:
    col = df["In_2.1.22-only"].astype("string").str.strip()
    mask = col.eq("---") | col.isna() | col.eq("")
    df = df[mask].copy()
df.head()

,precursor_protein,evidence_peptides_scans,combined_n_noncontained_le9,In_2.1.22-only
2,Q8N131,"[('LYIGCKMYY', 'MSV000096271', 19138, 'non_sta...",43,---
4,A2RRH5,"[('LSDIVIEKY', 'PXD029567', 641, 'IDENTIFIED',...",19,---
8,A6NNF4,"[('CLDTAQKNLY', 'MSV000080517', 1027, 'non_sta...",18,---
9,O15172,"[('TVISEEGIGCF', 'MSV000096271', 1763, 'non_st...",17,---
10,Q5UAW9,"[('PGESQESQGTPGELPST', 'MSV000080701', 16722, ...",16,---


In [34]:
df_pa = pd.read_csv("all_usi_Nov2025.xlsx - all_usi.tsv", sep="\t", usecols=["Dataset", "DemodPeptide"])
df_pa.head()

,Dataset,DemodPeptide
0,PXD006633,EIVMTQSPDTLSVSPGER
1,PXD006633,EIVMTQSPDTLSVSPGER
2,PXD006633,EIVMTQSPDTLSVSPGER
3,PXD006633,EIVMTQSPDTLSVSPGER
4,PXD006633,EIVMTQSPDTLSVSPGER


In [35]:
import ast
def _parse_evidence(peptide_entry):
    if peptide_entry is None:
        return []
    text = str(peptide_entry).strip()
    if not text:
        return []
    start = text.find('[')
    if start >= 0:
        normalized = text[start:]
    else:
        return []
    normalized = normalized.replace("nan", "None")
    if not normalized.endswith(']'):
        normalized = normalized + ']'
    try:
        return ast.literal_eval(normalized)
    except (ValueError, SyntaxError):
        return []

output a dataset_level.tsv that look like this, with the peptide dataset matching above: Dataset	num_retrieved_protein	specific_proteins
PXD019643	14	A0A0B4J271;A2RRH5;A4D0T7;A5LHX3;A6NNF4;O00270;O15172;Q3ZCN5;Q96KX1;Q99616;Q9BQJ4;Q9BTD3;Q9GZV3;Q9NQ39
PXD008333	10	A6NHN6;A8MPX8;B3SHH9;P0DTF9;Q1W4C9;Q30KQ8;Q6NXP0;Q8NG35;Q96J77;Q9NP94
PXD013649	7	A2RRH5;A6NNF4;O15172;Q6MZN7;Q6ZW05;Q8N6I4;Q9BRJ9
PXD004894	6	A6NNF4;Q6MZN7;Q6UX40;Q8N2M4;Q9BSJ1;Q9BTD3
PXD010154	6	A0A5B6;O95626;Q5VYV0;Q6UX40;Q6ZUT3;Q96DS6
PXD020079	6	A2RRH5;A4D0T7;A6NNF4;O15172;Q96N22;Q9BQJ4
PXD022150	6	A2RRH5;A4D0T7;A6NNF4;O15172;Q6UX40;Q9BTD3, but what i also want is that rank the num retrive protein from high to low, and if its in high, remove it in lower sets to form a union set of proteins

In [37]:
# Merge evidence peptides with dataset peptides
evidence_df = df
evidence_df["evidence_peptides_scans"] = evidence_df["evidence_peptides_scans"].apply(_parse_evidence)
evidence_df = evidence_df.explode("evidence_peptides_scans").dropna(subset=["evidence_peptides_scans"])
evidence_df[["peptide", "dataset","scan", "modification", "is_matched", "status"]] = pd.DataFrame(
    evidence_df["evidence_peptides_scans"].tolist(), index=evidence_df.index
)
merged = evidence_df.merge(
    df_pa,
    left_on="peptide",
    right_on="DemodPeptide",
    how="inner"
)

# Build initial per-dataset protein sets
dataset_proteins = (
    merged.groupby("Dataset")["precursor_protein"]
    .apply(lambda s: set(s.dropna()))
)

# Rank datasets by number of retrieved proteins (desc)
ranked = dataset_proteins.apply(len).sort_values(ascending=False)

# Build union-ranked sets: remove proteins already assigned to higher-ranked datasets
seen = set()
rows = []
while ranked.size > 0:
    # Pick the dataset with the most proteins
    ds = ranked.idxmax()
    proteins = dataset_proteins[ds] - seen
    seen |= proteins
    rows.append(
        {
            "Dataset": ds,
            "num_retrieved_protein": len(proteins),
            "specific_proteins": ";".join(sorted(proteins))
        }
    )
    # Remove the processed dataset from the ranking
    ranked = ranked.drop(ds)

result = pd.DataFrame(rows).sort_values(by="num_retrieved_protein", ascending=False)

# Save to TSV
result.to_csv("dataset_level.tsv", sep="\t", index=False)
result.head()

,Dataset,num_retrieved_protein,specific_proteins
0,PXD019643,41,A0A0B4J271;A0A7I2V2X1;A2RRH5;A4D0T7;A5LHX3;A6N...
1,PXD008333,22,A0A8V8TMC1;A0AAG2UWE3;A6NHN6;A6NNT2;A8MPX8;B3S...
2,PXD013649,17,A2RU14;A8MVM7;O75445;Q01726;Q6MZN7;Q6XD76;Q6ZM...
3,PXD010154,16,A0A0A6YYC5;A0A0K0K1C4;A0A1B0GUW7;A0A2R8YCJ5;A0...
4,MSV000084172,13,A0PJZ0;A6NEY8;O94777;P03901;P35638;P50406;Q158...
